In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.chdir('..')

In [ ]:
import torch
from src.config.swin_config import SwinConfig
from src.data.transformers_dataset import MelSpectrogramDataset
from src.training.swin_trainer import SwinTrainer
from src.utils.metrics import compute_metrics
from transformers import (
    SwinForImageClassification,
    AutoImageProcessor,
    TrainingArguments,
    EarlyStoppingCallback,
)

In [ ]:
device = SwinConfig.get_device()
image_processor = AutoImageProcessor.from_pretrained(SwinConfig.MODEL_ID)

preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [ ]:
model = SwinForImageClassification.from_pretrained(
    SwinConfig.MODEL_ID,
    num_labels              = SwinConfig.NUM_CLASSES,
    id2label                = SwinConfig.ID2LABEL,
    label2id                = SwinConfig.LABEL2ID,
    ignore_mismatched_sizes = True,
)
print(f"\n  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/113M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/233 [00:00<?, ?it/s]

SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.



  Trainable parameters: 27,520,892


In [8]:
print(f"  GPUs available: {torch.cuda.device_count()}")
model.to(device)

  GPUs available: 1


SwinForImageClassification(
  (swin): SwinModel(
    (embeddings): SwinEmbeddings(
      (patch_embeddings): SwinPatchEmbeddings(
        (projection): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): SwinEncoder(
      (layers): ModuleList(
        (0): SwinStage(
          (blocks): ModuleList(
            (0): SwinLayer(
              (layernorm_before): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
              (attention): SwinAttention(
                (self): SwinSelfAttention(
                  (query): Linear(in_features=96, out_features=96, bias=True)
                  (key): Linear(in_features=96, out_features=96, bias=True)
                  (value): Linear(in_features=96, out_features=96, bias=True)
                  (dropout): Dropout(p=0.0, inplace=False)
                )
                (output): SwinSelfOutput(
        

In [ ]:
train_ds = MelSpectrogramDataset(SwinConfig.TRAIN_PT, image_processor)
val_ds   = MelSpectrogramDataset(SwinConfig.VAL_PT,   image_processor)

  Loaded /kaggle/input/datasets/trieuvuongnguyen/for-preprocessed/processed/train.pt: 53866 samples
  Loaded /kaggle/input/datasets/trieuvuongnguyen/for-preprocessed/processed/val.pt: 10798 samples


In [ ]:
training_args = TrainingArguments(
    output_dir                  = SwinConfig.OUTPUT_DIR,
    num_train_epochs            = SwinConfig.EPOCHS,
    per_device_train_batch_size = SwinConfig.BATCH_SIZE,
    per_device_eval_batch_size  = SwinConfig.BATCH_SIZE * 2,
 
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "val_acc",
    greater_is_better           = True,
    save_total_limit            = 2,
 
    logging_dir                 = f"{SwinConfig.OUTPUT_DIR}/logs",
    logging_strategy            = "steps",
    logging_steps               = 50,
    report_to                   = "none",
 
    fp16                        = torch.cuda.is_available(),
    dataloader_num_workers      = 2,
    seed                        = SwinConfig.SEED,
    label_names                 = ["labels"],
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = SwinTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=5)],
)

In [12]:
print("\n  Starting Swin Transformer fine-tuning …\n")
trainer.train()


  Starting Swin Transformer fine-tuning …



Epoch,Training Loss,Validation Loss,Val Acc
1,0.009078,0.001487,0.999630
2,0.014345,0.002617,0.999537
3,0.000004,0.000664,0.999907
4,0.000007,0.001924,0.999722
5,0.000027,0.001092,0.999630
6,0.000006,0.001491,0.999722
7,0.001129,0.003012,0.999444
8,0.000004,0.002664,0.999537


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=26936, training_loss=0.011232814967219697, metrics={'train_runtime': 3300.6153, 'train_samples_per_second': 326.4, 'train_steps_per_second': 20.402, 'total_flos': 1.0711141621696954e+19, 'train_loss': 0.011232814967219697, 'epoch': 8.0})

In [ ]:
trainer.save_model(SwinConfig.FINAL_DIR)
image_processor.save_pretrained(SwinConfig.FINAL_DIR)
print(f"\n  Model saved {SwinConfig.FINAL_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  Model saved → ./swin-fakeaudio-final


In [14]:
print("\n  Final evaluation:")
print(trainer.evaluate())


  Final evaluation:


{'eval_loss': 0.0006644033128395677, 'eval_val_acc': 0.9999073902574551, 'eval_runtime': 25.3895, 'eval_samples_per_second': 425.294, 'eval_steps_per_second': 13.313, 'epoch': 8.0, 'train_acc': 0.0}
